# Model 3: BIST30 Stock Intraday Reaction Forecaster

## Executive Summary & Modeling Scope

The **BIST30 Stock Intraday Reaction Forecaster** predicts execution-aware intraday stock return percentages for BIST30 equities across three key post-opening reaction windows:
- **Window 2 (`first_reaction`, 10:30 – 11:30 TRT)**: Immediate momentum continuation or reversal after the opening session.
- **Window 3 (`midday_followup`, 11:30 – 14:30 TRT)**: Midday institutional accumulation and trend development.
- **Window 5 (`closing_session`, 16:00 – 18:15 TRT)**: End-of-day resolution and closing auction dynamics.

### Target Formulation
The regression target is defined as the execution-aware return %:
$$\text{Target Return}\% = \frac{\text{Window VWAP} - P_{\text{W1 reference}}}{P_{\text{W1 reference}}} \times 100$$

where $P_{\text{W1 reference}}$ is BofA's opening Buy VWAP (fallback: Market W1 VWAP).

> Zero lookahead guarantee: All input features use data available up to **10:30 TRT (end of Window 1)** on session $T$.

In [ ]:
import duckdb
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

from mdk_trading_oracle.core.db import DuckDBManager
from mdk_trading_oracle.core.config import get_settings
from mdk_trading_oracle.models.stock_reaction.features import StockReactionFeatureExtractor
from mdk_trading_oracle.models.stock_reaction.forecaster import StockReactionForecaster, StockReactionModelArena
from mdk_trading_oracle.models.stock_reaction.orchestrator import StockReactionOrchestrator
from mdk_trading_oracle.models.stock_reaction.models import (
    ReturnDirectionClassifier,
    ReturnThresholdProfile,
    StockReactionBayesianModel,
    StockReactionLightGBMModel,
    StockReactionNaivePersistenceModel,
    StockReactionRollingMeanModel,
    StockReactionXGBoostModel,
)

settings = get_settings()
db = DuckDBManager(read_only=True)
print(f"[OK] DuckDB Database: {settings.database_path}")
print(f"[OK] Stock Reaction Engine Ready")


## 1. Feature Extraction & Microstructure Clusters

Extract 8 quantitative feature clusters (47 features) for BIST30 equities (e.g. `AKBNK`).

In [ ]:
extractor = StockReactionFeatureExtractor(symbol="AKBNK", db=db)
df_feats = extractor.extract_features()
feat_cols = extractor.get_feature_columns()

print(f"[OK] Extracted {len(df_feats)} sessions for AKBNK across {len(feat_cols)} engineered feature columns.")
display(df_feats.select([
    "trade_date", "symbol", "feat_bofa_w1_net_flow_tl", "feat_bofa_w1_vol_share",
    "feat_comp_w1_net_flow_tl", "feat_w1_bofa_tra_contra_signal",
    "feat_bofa_t1_open_qty", "target_w2_return_pct", "target_w5_return_pct"
]).head(8).to_pandas())


## 2. Empirical Quantile Thresholds Profile

Inspect empirical return distributions ($P_{25}, P_{50}, P_{85}$) computed in `silver_stock_reaction_thresholds`.

In [ ]:
conn = db.get_connection()
df_thresh = conn.execute("""
    SELECT symbol, window_name, up_p25_pct, up_p50_pct, up_p85_pct, down_p85_pct, total_sessions
    FROM silver_stock_reaction_thresholds
    WHERE symbol IN ('AKBNK', 'GARAN', 'THYAO', 'ASELS', 'KCHOL')
    ORDER BY symbol, window_name;
""").pl()
display(df_thresh.to_pandas())


## 3. Candidate Model Arena & Tournament

Run expanding-window walk-forward validation across candidate paradigms for `AKBNK` (Window 2: `first_reaction`).

In [ ]:
forecaster_w2 = StockReactionForecaster(symbol="AKBNK", window="w2", db=db)
X_train, y_train = forecaster_w2._prepare_training_data()
thresholds = forecaster_w2._load_thresholds()

arena = StockReactionModelArena(symbol="AKBNK", window="w2", thresholds=thresholds)
scoreboard, champion = arena.run_tournament(X_train, y_train, min_train_samples=5)

display(scoreboard[["Model", "hit_rate_pct", "picp_90_pct", "mae_pct", "rmse_pct", "sample_size"]])
print(f"Champion Model: {champion.model_name}")


## 4. Live Upcoming Session Signal Card ($T+1$)

Query the latest active forecasts from the Gold tables (`gold_bofa_stock_reaction_*_forecasts`).

In [ ]:
orchestrator = StockReactionOrchestrator(db=db)
df_w2_live = orchestrator.get_latest_forecasts(window="w2")
if df_w2_live is not None and not df_w2_live.is_empty():
    display(df_w2_live.to_pandas().head(10))
else:
    print("[INFO] No live forecasts currently active. Run pipeline to generate live inference.")


## 5. Performance Ledger & Backtest Verification

Inspect historical reconciled track records and walk-forward simulation ledgers in DuckDB.

In [ ]:
conn = db.get_connection()
perf_w2 = conn.execute("""
    SELECT symbol, COUNT(*) as n, AVG(CAST(is_direction_hit AS INTEGER))*100 as hit_rate_pct,
           AVG(absolute_error_pct) as avg_mae_pct, AVG(CAST(is_inside_90_ci AS INTEGER))*100 as picp_90_pct
    FROM gold_bofa_stock_reaction_w2_performance
    WHERE actual_return_pct IS NOT NULL
    GROUP BY symbol
    ORDER BY hit_rate_pct DESC;
""").pl()
display(perf_w2.to_pandas())


## 6. DuckDB Gold Layer Tables Overview

Verify the 9 Model 3 Gold tables.

In [ ]:
tables = conn.execute("""
    SHOW TABLES;
""").fetchall()
stock_reaction_tables = [t[0] for t in tables if "stock_reaction" in t[0]]
print(f"Stock Reaction Gold & Silver Tables ({len(stock_reaction_tables)}):\n")
for tbl in sorted(stock_reaction_tables):
    cnt = conn.execute(f"SELECT COUNT(*) FROM {tbl};").fetchone()[0]
    print(f"  - {tbl:<45}: {cnt:>6,} rows")
